## Features Used in ScholarAI Project
The following GenAI features from the list were successfully implemented:

* **Few-shot prompting** – Custom prompt templates were used to guide the summarization and Q&A responses.
* **Document understanding** – Academic texts were split, embedded, and processed using LangChain's document tools.
* **Embeddings** – Text chunks were converted into embeddings using `GoogleGenerativeAIEmbeddings`.
* **Retrieval-Augmented Generation (RAG)** – The core system retrieves relevant documents before generating answers.
* **Vector search/vector store/vector database** – Used `InMemoryVectorStore` for similarity-based retrieval of academic content.

## Diagram for ScholarAI

![ScholarAI Diagram](https://i.postimg.cc/KcMBr9Qv/Diagram-for-Scholar-AI.png)

## Notebook Imports

In [ ]:
# Remove conflicting packages from the Kaggle base environment.
!pip uninstall -qqy kfp jupyterlab libpysal thinc spacy fastai ydata-profiling google-cloud-bigquery google-generativeai
# Install langgraph and the packages used in this lab.
!pip install -qU 'langgraph==0.3.21' 'langchain-google-genai==2.1.2' 'langgraph-prebuilt==0.1.7'

In [ ]:
!pip install -qU langchain langchain-community

In [ ]:
!pip install -qU langchain-core

In [ ]:
# General Python Libraries
import os
import pandas as pd
from typing_extensions import List, TypedDict

# Kaggle Secrets
from kaggle_secrets import UserSecretsClient

# LangChain Core
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.vectorstores import InMemoryVectorStore
from langchain.chains.llm import LLMChain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain import FewShotPromptTemplate

# LangChain Utilities
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader

# LangChain + Gemini
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings
)

# Gemini Native SDK (optional)
from google import genai
from google.genai import types

In [ ]:
GOOGLE_API_KEY = UserSecretsClient().get_secret("GOOGLE_API_KEY")
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

client = genai.Client(api_key=GOOGLE_API_KEY)

## 📝 Document Summarizer

The Document Summarizer enables students to upload learning materials (such as class notes, articles, or textbooks) and instantly receive a concise summary.

In [ ]:
data1_path = '/kaggle/input/arxiv-paper-abstracts/arxiv_data.csv'
data2_path = '/kaggle/input/arxiv-paper-abstracts/arxiv_data_210930-054931.csv'

df1 = pd.read_csv(data1_path)
df1.head()

In [ ]:
df2 = pd.read_csv(data2_path)
df2['summaries'] = df2['abstracts']
df2 = df2.drop("abstracts", axis='columns')
df2.tail()

In [ ]:
df = pd.concat([df1, df2], ignore_index=True)
df.head()

In [ ]:
df = df.sample(frac=0.01, random_state=42).reset_index().drop("index", axis='columns')
df.shape

In [ ]:
def get_text(dataframe: pd.DataFrame, max_count: int = 2) -> str:
    entries = []
    for i, row in dataframe[['titles', 'summaries']].dropna().head(max_count).iterrows():
        entry = f"Title: {row['titles']}\nSummary: {row['summaries']}"
        entries.append(entry)
    return "\n\n".join(entries)

In [ ]:
def text_summarizer(text):
    # Step 1: Split text into chunks
    text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
        chunk_size=500,
        chunk_overlap=50,
    )
    docs = [Document(page_content=text)]
    split_docs = text_splitter.split_documents(docs)

    # Step 2: Set up Gemini via LangChain GoogleGenerativeAI
    llm = GoogleGenerativeAI(
        model="models/gemini-2.0-flash",
        google_api_key=GOOGLE_API_KEY,
        temperature=0.1
    )

    # Step 3: Use context as the input variable
    prompt = PromptTemplate(
        input_variables=["context"],
        template=(
            "You are an expert academic summarizer.\n"
            "Summarize the following academic research papers into concise paragraphs:\n\n"
            "{context}\n\n"
            "Summary:"
        )
    )

    # Step 4: Create the chain
    chain = create_stuff_documents_chain(llm, prompt)

    # Step 5: Run the chain
    result = chain.invoke({"context": split_docs})

    return result

In [ ]:
text = get_text(df)
text

In [ ]:
summary = text_summarizer(text=text)
print(summary)

## 🔍 Personalized Q&A (RAG system)
This tool enables students to ask questions grounded in their own study materials using a Retrieval-Augmented Generation (RAG) pipeline. 

In [ ]:
text

In [ ]:
# Splitting documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,  # chunk size (characters)
    chunk_overlap=100,  # chunk overlap (characters)
    add_start_index=True,  # track index in original document
)

docs = [Document(page_content=text)]
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

In [ ]:
vector_store = InMemoryVectorStore(embeddings)
vector_store.add_documents(all_splits)

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="models/gemini-2.0-flash",
    temperature=0.2,
    google_api_key=GOOGLE_API_KEY
)

In [ ]:
template = """Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
Use 4-5 sentences maximum and keep the answer as concise as possible. answer in Japanese.

{context}

Question: {question}

Helpful Answer:"""

prompt = PromptTemplate(
    input_variables=["question", "context"],
    template=template
)

In [ ]:
class State(TypedDict):
    question: str
    context: List[Document]
    answer: str

In [ ]:
def retrieve(state: State):
    retrieved_docs = vector_store.similarity_search(state["question"])
    return {
        "question": state["question"],
        "context": retrieved_docs
    }


def generate(state: State):
    docs_content = "\n\n".join(doc.page_content for doc in state["context"])
    formatted_prompt = prompt.format(question=state["question"], context=docs_content)
    response = llm.invoke(formatted_prompt)
    return {"answer": response.content}

In [ ]:
# Wrap your custom functions
retrieve_runnable = RunnableLambda(retrieve)
generate_runnable = RunnableLambda(generate)

# Chain them
rag_chain = retrieve_runnable | generate_runnable

In [ ]:
def ask_rag_question(question: str) -> str:
    state = {"question": question, "context": [], "answer": ""}
    return rag_chain.invoke(state)["answer"]

In [ ]:
question = "MORAN is stands for?"
answer = ask_rag_question(question)
print(answer)

In [ ]:
question = "MORANは何を表していますか?"
answer = ask_rag_question(question)
print(answer)

# Future Work
To improve and expand the current system, the following future enhancements are planned:

* **PDF Upload Support** – Allow users to upload their own documents for Q&A.
* **Persistent Vector Store** – Replace in-memory storage with FAISS or Chroma for scalability.
* **Better Prompts** – Use more dynamic prompts for improved answer quality.
* **User Feedback** – Add ratings or comments to evaluate and refine responses.
* **LangGraph Integration** – Explore more advanced workflows and multi-turn reasoning.
* **Web Interface** – Deploy with Streamlit or Gradio for easier user interaction.

# Thank You!

This project was built as part of the GenAI Capstone.

I’m proud of how far I’ve come. I am excited to grow further.

Thank you for reviewing ScholarAI!